In [0]:
dbutils.widgets.text("catalog_name",  "dbw_chinook_team", "Catalog")
dbutils.widgets.text("silver_schema", "chinook_silver",   "Silver")
dbutils.widgets.text("gold_schema",   "chinook_gold",     "Gold")

In [0]:
from pyspark.sql.functions import (
    col, concat_ws, md5, lit, to_date,
    current_timestamp, date_format,
    year, month, quarter, dayofmonth,
    dayofweek, weekofyear, coalesce, when
)
from delta.tables import DeltaTable

CATALOG = dbutils.widgets.get("catalog_name")
SILVER  = f"{CATALOG}.{dbutils.widgets.get('silver_schema')}"
GOLD    = f"{CATALOG}.{dbutils.widgets.get('gold_schema')}"

print(f"✅ Catalog : {CATALOG}")
print(f"✅ Silver  : {SILVER}")
print(f"✅ Gold    : {GOLD}")

In [0]:
print("Building DIM_DATE...")

date_dim = spark.sql("""
    SELECT
        CAST(date_format(d,'yyyyMMdd') AS INT) AS date_sk,
        d                                       AS full_date,
        year(d)                                 AS year,
        quarter(d)                              AS quarter,
        month(d)                                AS month,
        date_format(d,'MMMM')                  AS month_name,
        dayofmonth(d)                           AS day,
        dayofweek(d)                            AS day_of_week,
        date_format(d,'EEEE')                  AS day_name,
        weekofyear(d)                           AS week_of_year
    FROM (
        SELECT explode(sequence(
            to_date('2000-01-01'),
            to_date('2030-12-31'),
            interval 1 day
        )) AS d
    )
""")

date_dim.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable(f"{GOLD}.dim_date")
print(f"✅ DIM_DATE: {date_dim.count()} rows")

In [0]:
print("Building DIM_CUSTOMER (SCD Type 2)...")

customer_silver = spark.table(f"{SILVER}.customer")

new_customers = customer_silver.select(
    col("CustomerId").alias("customer_id"),
    concat_ws(" ", col("FirstName"),
              col("LastName")).alias("customer_name"),
    col("FirstName").alias("first_name"),
    col("LastName").alias("last_name"),
    col("Email").alias("email"),
    col("Phone").alias("phone"),
    col("Address").alias("address"),
    col("City").alias("city"),
    col("State").alias("state"),
    col("Country").alias("country"),
    col("PostalCode").alias("postal_code"),
    col("Company").alias("company"),
    col("SupportRepId").alias("support_rep_id")
).withColumn("hash_value",
    md5(concat_ws("||",
        coalesce(col("first_name"),  lit("")),
        coalesce(col("last_name"),   lit("")),
        coalesce(col("email"),       lit("")),
        coalesce(col("phone"),       lit("")),
        coalesce(col("address"),     lit("")),
        coalesce(col("city"),        lit("")),
        coalesce(col("state"),       lit("")),
        coalesce(col("country"),     lit("")),
        coalesce(col("postal_code"), lit(""))
    ))
).withColumn("customer_key",
    md5(col("customer_id").cast("string"))) \
 .withColumn("effective_start_date", current_timestamp()) \
 .withColumn("effective_end_date",
    lit(None).cast("timestamp")) \
 .withColumn("is_current", lit(True))

if not spark.catalog.tableExists(f"{GOLD}.dim_customer"):
    new_customers.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema","true") \
        .saveAsTable(f"{GOLD}.dim_customer")
    print(f"✅ DIM_CUSTOMER created: {new_customers.count()} rows")
else:
    existing_df = spark.table(f"{GOLD}.dim_customer") \
        .filter("is_current = true")

    changes = new_customers.join(
        existing_df.select(
            "customer_id",
            col("hash_value").alias("existing_hash")
        ),
        "customer_id", "left"
    ).filter(
        col("existing_hash").isNull() |
        (col("hash_value") != col("existing_hash"))
    ).drop("existing_hash")

    if changes.count() > 0:
        existing = DeltaTable.forName(spark, f"{GOLD}.dim_customer")
        changed_ids = [r.customer_id for r in changes.collect()]

        existing.update(
            condition=col("customer_id").isin(changed_ids)
                    & col("is_current"),
            set={
                "effective_end_date": current_timestamp(),
                "is_current": lit(False)
            }
        )
        changes.write.format("delta") \
            .mode("append") \
            .saveAsTable(f"{GOLD}.dim_customer")
        print(f"✅ DIM_CUSTOMER: {changes.count()} new/changed records")
    else:
        print("✅ DIM_CUSTOMER: no changes detected")

total = spark.table(f"{GOLD}.dim_customer").count()
print(f"   Total rows: {total}")

In [0]:
print("Building DIM_TRACK...")

track     = spark.table(f"{SILVER}.track")
album     = spark.table(f"{SILVER}.album")
artist    = spark.table(f"{SILVER}.artist")
genre     = spark.table(f"{SILVER}.genre")
mediatype = spark.table(f"{SILVER}.mediatype")

track_dim = track \
    .join(album,     "AlbumId",     "left") \
    .join(artist,    "ArtistId",    "left") \
    .join(genre,     "GenreId",     "left") \
    .join(mediatype, "MediaTypeId", "left") \
    .select(
        track["TrackId"].alias("track_nk"),
        md5(track["TrackId"].cast("string")).alias("track_key"),
        track["Name"].alias("track_name"),
        album["Title"].alias("album_title"),
        artist["Name"].alias("artist_name"),
        genre["Name"].alias("genre_name"),
        mediatype["Name"].alias("media_type_name"),
        track["Composer"].alias("composer"),
        track["Milliseconds"].alias("duration_ms"),
        track["Bytes"].alias("file_size_bytes"),
        track["UnitPrice"].cast("decimal(10,2)").alias("unit_price")
    )

track_dim.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable(f"{GOLD}.dim_track")
print(f"✅ DIM_TRACK: {track_dim.count()} rows")

In [0]:
print("Building DIM_EMPLOYEE...")

employee = spark.table(f"{SILVER}.employee")

employee_dim = employee.select(
    col("EmployeeId").alias("employee_nk"),
    md5(col("EmployeeId").cast("string")).alias("employee_key"),
    concat_ws(" ", col("FirstName"),
              col("LastName")).alias("employee_name"),
    col("Title").alias("title"),
    col("ReportsTo").alias("reports_to_nk"),
    col("BirthDate").alias("birth_date"),
    col("HireDate").alias("hire_date"),
    col("City").alias("city"),
    col("State").alias("state"),
    col("Country").alias("country"),
    col("Email").alias("email")
)

employee_dim.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable(f"{GOLD}.dim_employee")
print(f"✅ DIM_EMPLOYEE: {employee_dim.count()} rows")

In [0]:
print("Building FACT_SALES...")

invoice      = spark.table(f"{SILVER}.invoice")
invoiceline  = spark.table(f"{SILVER}.invoiceline")
dim_customer = spark.table(f"{GOLD}.dim_customer").filter("is_current = true")
dim_track    = spark.table(f"{GOLD}.dim_track")
dim_employee = spark.table(f"{GOLD}.dim_employee")
dim_date     = spark.table(f"{GOLD}.dim_date")
customer_sv  = spark.table(f"{SILVER}.customer")

invoice_keyed = invoice.withColumn(
    "date_sk",
    date_format(col("InvoiceDate"), "yyyyMMdd").cast("int")
)

sales_fact = invoiceline \
    .join(invoice_keyed, "InvoiceId", "inner") \
    .join(
        dim_customer.select("customer_id","customer_key"),
        invoice_keyed["CustomerId"] == dim_customer["customer_id"],
        "left"
    ) \
    .join(
        dim_track.select("track_nk","track_key"),
        invoiceline["TrackId"] == dim_track["track_nk"],
        "left"
    ) \
    .join(
        dim_date.select("date_sk","full_date"),
        invoice_keyed["date_sk"] == dim_date["date_sk"],
        "left"
    ) \
    .join(
        customer_sv.select("CustomerId","SupportRepId"),
        "CustomerId", "left"
    ) \
    .join(
        dim_employee.select("employee_nk","employee_key"),
        customer_sv["SupportRepId"] == dim_employee["employee_nk"],
        "left"
    ) \
    .select(
        invoiceline["InvoiceLineId"].alias("invoice_line_id"),
        invoice_keyed["InvoiceId"].alias("invoice_id"),
        col("customer_key"),
        col("track_key"),
        invoice_keyed["date_sk"],
        col("employee_key"),
        invoice_keyed["BillingCountry"].alias("billing_country"),
        invoice_keyed["BillingState"].alias("billing_state"),
        invoiceline["Quantity"].alias("quantity"),
        invoiceline["UnitPrice"].cast("decimal(10,2)").alias("unit_price"),
        (invoiceline["Quantity"] * invoiceline["UnitPrice"])
            .cast("decimal(10,2)").alias("line_total"),
        invoice_keyed["Total"].cast("decimal(10,2)").alias("invoice_total")
    )

sales_fact.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable(f"{GOLD}.fact_sales")
print(f"✅ FACT_SALES: {sales_fact.count()} rows")

In [0]:
print("Building FACT_SALES_CUSTOMER_AGG...")

from pyspark.sql.functions import sum, count, avg, max, min

fact_sales   = spark.table(f"{GOLD}.fact_sales")
dim_customer = spark.table(f"{GOLD}.dim_customer").filter("is_current = true")

agg_fact = fact_sales \
    .join(
        dim_customer.select(
            "customer_key","customer_name",
            "country","city"
        ),
        "customer_key", "left"
    ) \
    .groupBy("customer_key","customer_name","country","city") \
    .agg(
        count("invoice_line_id").alias("total_line_items"),
        sum("line_total").cast("decimal(12,2)").alias("total_revenue"),
        avg("line_total").cast("decimal(10,2)").alias("avg_order_value"),
        count("invoice_id").alias("total_invoices"),
        max("line_total").cast("decimal(10,2)").alias("max_order_value"),
        min("line_total").cast("decimal(10,2)").alias("min_order_value")
    )

agg_fact.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable(f"{GOLD}.fact_sales_customer_agg")
print(f"✅ FACT_SALES_CUSTOMER_AGG: {agg_fact.count()} rows")

In [0]:
print("\n=== GOLD LAYER VALIDATION ===")

gold_tables = [
    "dim_date", "dim_customer", "dim_track",
    "dim_employee", "fact_sales",
    "fact_sales_customer_agg"
]

for tbl in gold_tables:
    count = spark.table(f"{GOLD}.{tbl}").count()
    print(f"  ✅ {tbl:<30} {count:>6} rows")

# FK null check on fact_sales
fact = spark.table(f"{GOLD}.fact_sales")
print(f"\n=== FK VALIDATION ===")
print(f"  Null customer_key : {fact.filter(col('customer_key').isNull()).count()}")
print(f"  Null track_key    : {fact.filter(col('track_key').isNull()).count()}")
print(f"  Null date_sk      : {fact.filter(col('date_sk').isNull()).count()}")
print(f"  Null employee_key : {fact.filter(col('employee_key').isNull()).count()}")

print("\n GOLD LAYER COMPLETE!")